In [1]:
!git clone https://github.com/rashibharti28/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion.git

Cloning into 'BERT-Quantization-PTQ-QAT-on-dair-ai-emotion'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 94 (delta 38), reused 23 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 4.62 MiB | 10.23 MiB/s, done.
Resolving deltas: 100% (38/38), done.
Filtering content: 100% (5/5), 1.95 GiB | 41.22 MiB/s, done.


In [ ]:
# Install deps in Colab first
!pip install -U transformers datasets accelerate peft bitsandbytes evaluate safetensors




In [ ]:
#!/usr/bin/env python3


import sys
if 'ipykernel' in sys.modules: sys.argv = sys.argv[:1]

import argparse, traceback, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, BitsAndBytesConfig, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate

def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tune DistilBERT with QLoRA on dair-ai/emotion", add_help=True)
    parser.add_argument("--model_name", type=str, default="distilbert-base-uncased")
    parser.add_argument("--dataset_name", type=str, default="dair-ai/emotion")
    parser.add_argument("--output_dir", type=str, default="./distilbert-qlora-emotion")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--per_device_train_batch_size", type=int, default=16)
    parser.add_argument("--per_device_eval_batch_size", type=int, default=64)
    parser.add_argument("--learning_rate", type=float, default=2e-4)
    parser.add_argument("--weight_decay", type=float, default=0.01)   # <-- added
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--lora_r", type=int, default=16, help="LoRA rank r (try 8,16,32)")
    parser.add_argument("--lora_alpha", type=int, default=32, help="LoRA alpha scaling")
    parser.add_argument("--lora_dropout", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--save_total_limit", type=int, default=2)
    parser.add_argument("--device_map", type=str, default="auto", help="device map for model placement; use 'auto' or 'cuda'/'cpu'")
    parser.add_argument("--use_small_subset", action="store_true", help="use small subset for quick tests")
    parser.add_argument("--no_quant", action="store_true", help="Skip 4-bit quantization and load full precision model")
    parser.add_argument("--force_quant", action="store_true", help="Force trying quantized load (unsafe).")
    args, unknown = parser.parse_known_args()
    return args


def safe_load_model(model_name, num_labels, prefer_quant=False):
    cfg = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
    # If user didn't request quant, skip it.
    if not prefer_quant:
        print("[SAFE LOAD] Skipping quantization (no_quant). Loading full-precision model with low_cpu_mem_usage.")
        return AutoModelForSequenceClassification.from_pretrained(model_name, config=cfg, device_map=("auto" if torch.cuda.is_available() else None), low_cpu_mem_usage=True), False

    # If prefer_quant True (user forced), attempt quantized loads with fallbacks
    try:
        print("[TRY] Attempting 4-bit quantized load (device_map='auto')...")
        bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, config=cfg, device_map=("auto" if torch.cuda.is_available() else None), quantization_config=bnb_cfg, low_cpu_mem_usage=True)
        print("[OK] Quantized loaded (auto).")
        return model, True
    except Exception:
        tb = traceback.format_exc()
        print("[WARN] Quantized (auto) failed:\n", tb)

    try:
        print("[TRY] Attempting quantized load to CPU then move to device...")
        bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, config=cfg, device_map={"": "cpu"}, quantization_config=bnb_cfg, low_cpu_mem_usage=True)
        if torch.cuda.is_available():
            try:
                model.to("cuda")
            except Exception as e:
                print("[WARN] Manual .to('cuda') for quant model raised (nonfatal):", e)
        print("[OK] Quantized loaded via CPU fallback.")
        return model, True
    except Exception:
        tb2 = traceback.format_exc()
        print("[WARN] Quantized CPU-load failed:\n", tb2)

    # Final fallback: non-quantized
    print("[FALLBACK] Loading non-quantized model (full precision).")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=cfg, device_map=("auto" if torch.cuda.is_available() else None), low_cpu_mem_usage=True)
    return model, False

def main():
    args = parse_args()
    torch.manual_seed(args.seed)

    print("Loading dataset:", args.dataset_name)
    ds = load_dataset(args.dataset_name)
    label_list = ds["train"].features["label"].names if hasattr(ds["train"].features["label"], "names") else None
    num_labels = len(label_list) if label_list else len(set(ds["train"]["label"]))
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)

    def prep(batch):
        enc = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=args.max_length)
        enc["labels"] = batch["label"]
        return enc

    ds = ds.map(prep, batched=True, remove_columns=ds["train"].column_names)
    train_ds = ds["train"]
    val_ds = ds["validation"] if "validation" in ds else ds["test"]

    if args.use_small_subset:
        train_ds = train_ds.shuffle(seed=args.seed).select(range(min(500, len(train_ds))))
        val_ds = val_ds.shuffle(seed=args.seed).select(range(min(200, len(val_ds))))

    # Decide whether we attempt quant: prefer quant only if force_quant True and no_quant not given
    prefer_quant = args.force_quant and (not args.no_quant)

    model, used_quant = safe_load_model(args.model_name, num_labels, prefer_quant=prefer_quant)
    print("Used quantized model?" , used_quant)

    if used_quant:
        try:
            model = prepare_model_for_kbit_training(model)
        except Exception as e:
            print("[WARN] prepare_model_for_kbit_training failed:", e)

    # attach LoRA
    target_modules = ["q_lin", "k_lin", "v_lin", "out_lin", "lin"]
    lconf = LoraConfig(r=args.lora_r, lora_alpha=args.lora_alpha, target_modules=target_modules, lora_dropout=args.lora_dropout, bias="none", task_type="SEQ_CLS")
    model = get_peft_model(model, lconf)
    # print trainable params
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable}/{total} ({100*trainable/total:.6f}%)")

    data_collator = DataCollatorWithPadding(tokenizer)
    acc = evaluate.load("accuracy"); f1 = evaluate.load("f1")
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = logits.argmax(-1)
        return {"accuracy": acc.compute(predictions=preds, references=labels)["accuracy"], "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]}

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        gradient_accumulation_steps=max(1, 8 // max(1, args.per_device_train_batch_size)),
        num_train_epochs=args.epochs,
        fp16=torch.cuda.is_available(),
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        eval_strategy="epoch",   # evaluate once per epoch
        save_strategy="epoch",         # must match evaluation_strategy when load_best_model_at_end=True
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=args.save_total_limit,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        remove_unused_columns=True,
        dataloader_num_workers=2,
        report_to=["none"]
    )



    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)

    # Wrap training to auto-retry once on OOM with smaller batch
    try:
        trainer.train()
    except RuntimeError as e:
        msg = str(e).lower()
        if "out of memory" in msg or "cuda out of memory" in msg:
            print("[OOM] Detected OOM. Retrying with half batch size...")
            try:
                new_bs = max(1, args.per_device_train_batch_size // 2)
                trainer.args.per_device_train_batch_size = new_bs
                trainer.args.gradient_accumulation_steps = max(1, trainer.args.gradient_accumulation_steps * 2)
                print(f"[OOM] New per-device batch size: {new_bs}, gradient_accumulation_steps: {trainer.args.gradient_accumulation_steps}")
                trainer.train()
            except Exception as e2:
                print("[ERROR] Retry after OOM failed. Exiting. Error:", e2)
                raise
        else:
            raise

    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print("Saved to", args.output_dir)

if __name__ == "__main__":
    main()


Loading dataset: dair-ai/emotion


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[SAFE LOAD] Skipping quantization (no_quant). Loading full-precision model with low_cpu_mem_usage.
Used quantized model? False
Trainable params: 1185030/68143116 (1.739031%)


/tmp/ipython-input-4059554608.py:159: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.284000,0.207627,0.921500,0.894619
2,0.180000,0.191348,0.928000,0.900825
3,0.131800,0.172657,0.933000,0.909396


Saved to ./distilbert-qlora-emotion
